# Train adapter synth2 (A100)

Adapter thứ ba của vote, train trên `data_full_e1` (TRAIN) + `synth_v2_rows` (769 task synth v2, bản gold + pseudo-ASR).
LoRA r16/alpha 32, 2 epoch, lr 1e-4 trên base bf16 (như p1, cũng là bản vLLM phục vụ).

Trên Drive, thư mục `lifemate_synth2/` cần có:
- `bundle.zip`: code + `data_full_e1` + `synth_v2_rows` (tạo bằng `scripts/train/make_synth2_bundle.sh`)

Checkpoint được ghi thẳng vào `lifemate_synth2/run/` mỗi 150 bước. Nếu runtime bị ngắt, chạy lại tất cả các cell thì sẽ train tiếp từ checkpoint cuối.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
D = Path('/content/drive/MyDrive/lifemate_synth2')
assert (D/'bundle.zip').exists(), 'thiếu bundle.zip'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf repo && mkdir repo && cd repo && unzip -q {D}/bundle.zip
%cd /content/repo
!pip -q install "unsloth==2026.10.11" "unsloth_zoo==2026.10.11" "trl==1.15.0" "peft==0.21.2" "transformers==5.17.0" "torch==2.14.1" "accelerate==1.15.0" "datasets==4.8.5" "bitsandbytes==0.50.2" "huggingface_hub"
!pip -q install -e . --no-deps

In [ ]:
# Base Qwen3.5-4B, đúng revision mà vLLM phục vụ (scripts/setup/get_release_inference_models.py)
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3.5-4B', revision='851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', local_dir='models/qwen3.5-4b', max_workers=4)

In [ ]:
# data_full_e1 + synth v2: mỗi dòng synth một lần
import json, shutil
out = Path('outputs/sft/data_full_synth2'); shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)
counts = {}
for part in ('train', 'dev'):
    rows = []
    for src in ('outputs/sft/data_full_e1', 'outputs/sft/synth_v2_rows'):
        rows += [r for r in Path(f'{src}/{part}.jsonl').read_text(encoding='utf-8').splitlines() if r.strip()]
    (out/f'{part}.jsonl').write_text('\n'.join(rows) + '\n', encoding='utf-8')
    counts[part] = len(rows)
(out/'manifest.json').write_text(json.dumps({'sources': ['data_full_e1', 'synth_v2_rows'], 'counts': counts}, indent=2))
print(counts)   # mong đợi {'train': 4144, 'dev': 484}

In [ ]:
# Train. --resume: lấy checkpoint mới nhất trong run/ nếu có.
RUN = D/'run'
!PYTHONUNBUFFERED=1 python scripts/train/train_agent_sft.py --data-dir outputs/sft/data_full_synth2 --out {RUN} --model models/qwen3.5-4b --epochs 2 --lr 1e-4 --save-steps 150 --bf16-base --grad-ckpt on --resume

In [ ]:
# Adapter cuối: run/adapter (adapter_config.json + adapter_model.safetensors). Chép ra để notebook vLLM nạp làm lifemate_synth.
final = D/'llm-lora-synth2'; shutil.rmtree(final, ignore_errors=True)
shutil.copytree(RUN/'adapter', final)
print(sorted(p.name for p in final.iterdir()))